# Hindi Pipecat Voicebot Colab Runner

Run this notebook on a GPU runtime. It installs Pipecat, faster-whisper, F5-TTS, llama.cpp, starts the backend/frontend, and opens Ngrok tunnels.

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path('/content/HB')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print('Project dir:', PROJECT_DIR)

## Configure secrets and model paths

Set `NGROK_AUTHTOKEN`. If the Orato or IndicF5 Hugging Face repos ask for auth, also set `HF_TOKEN`. Provide either a local `QWEN_GGUF_PATH` or a direct `QWEN_GGUF_URL`.

In [ ]:
NGROK_AUTHTOKEN = ''  # paste token or set via Colab secrets
HF_TOKEN = ''          # optional; needed for gated Hugging Face model access
QWEN_GGUF_URL = ''     # optional direct download URL
QWEN_GGUF_PATH = '/content/models/qwen3-8b-q4_k_m.gguf'

if NGROK_AUTHTOKEN:
    os.environ['NGROK_AUTHTOKEN'] = NGROK_AUTHTOKEN
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['QWEN_GGUF_PATH'] = QWEN_GGUF_PATH
os.environ['LLAMA_BASE_URL'] = 'http://127.0.0.1:8080/v1'
os.environ['LLAMA_MODEL'] = 'qwen3-8b'
os.environ['WHISPER_MODEL'] = 'small'
os.environ['WHISPER_DEVICE'] = 'cuda'
os.environ['WHISPER_COMPUTE_TYPE'] = 'float16'
os.environ['ORATO_MODEL_ID'] = 'tryorato/orato-tts-hindi-v1'
os.environ['ORATO_VOICE'] = 'female'

In [ ]:
!nvidia-smi
!apt-get update -y >/dev/null
!apt-get install -y build-essential cmake ninja-build ffmpeg nodejs npm >/dev/null
!pip install -U pip >/dev/null
!pip install -r /content/HB/requirements.txt
!pip install git+https://github.com/SWivid/F5-TTS.git

## Build llama.cpp

In [ ]:
%cd /content
!test -d llama.cpp || git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
%cd /content/llama.cpp
!cmake -B build -DGGML_CUDA=ON -DLLAMA_CURL=ON
!cmake --build build --config Release -j 2
os.environ['LLAMA_SERVER_BIN'] = '/content/llama.cpp/build/bin/llama-server'

## Download Qwen GGUF if a URL was provided

In [ ]:
from pathlib import Path
Path(QWEN_GGUF_PATH).parent.mkdir(parents=True, exist_ok=True)
if QWEN_GGUF_URL and not Path(QWEN_GGUF_PATH).exists():
    !wget -O {QWEN_GGUF_PATH} {QWEN_GGUF_URL}
print('Qwen GGUF exists:', Path(QWEN_GGUF_PATH).exists(), QWEN_GGUF_PATH)

## Install frontend dependencies

In [ ]:
%cd /content/HB/frontend
!npm install

## Start services and Ngrok tunnels

This cell blocks while the services are running. Open the printed Voice console URL in a new browser tab.

In [ ]:
%cd /content/HB
!python scripts/colab_start.py